Parto da Documents/problemi_deduplicati.txt (già ordinato dal problema con più fonti fuse al meno). Per ogni problema con più di una fonte, chiedo a un LLM locale (LM Studio) di unificare le occorrenze in un'unica scheda di sintesi. I problemi con una sola fonte li lascio invariati (solo ripulendo l'intestazione). Alla fine salvo ogni problema come file .md separato in Documents/singoli_dedup/

# Parsing di problemi_deduplicati.txt

In [ ]:
import os
import re

INPUT_PATH = os.path.join("..", "Documents", "problemi_deduplicati.txt")

HEADER_RE = re.compile(r"^===\s*PROBLEMA\s+(\d+)\s*\((\d+)\s*fonte/i fusa/e\)\s*===") #becca il problema e il numero di fonti
FONTE_RE = re.compile(r"--- Fonte: (?P<file>.+?) \((?P<year>\d+)\) ---\n")# becca il nome del file e l'anno della fonte
BULLET_DATE_RE = re.compile(r"^•\s*\[[^\]]+\]\s*—\s*")# trova il coso pntato prodotto da bullet point e data


def parse_sources(body):
    matches = list(FONTE_RE.finditer(body))
    sources = []
    for idx, m in enumerate(matches):
        start = m.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(body)
        text = body[start:end].strip("\n")
        sources.append({
            "source_file": m.group("file"),
            "year": m.group("year"),
            "text": text,
        })
    return sources


def parse_file(path):
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    raw_blocks = re.split(r"\n\s*-%-\s*\n", content) #divido di -%-
    raw_blocks = [b.strip("\n") for b in raw_blocks if b.strip()]

    blocks = []
    for b in raw_blocks:
        lines = b.split("\n")
        m = HEADER_RE.match(lines[0].strip())
        if not m:
            print(f"ATTENZIONE: blocco senza header valido, inizio: {lines[0][:80]!r}")
            continue
        number = int(m.group(1))
        body = "\n".join(lines[1:])
        sources = parse_sources(body)
        blocks.append({"number": number, "n_sources": len(sources), "sources": sources})
    return blocks


blocks = parse_file(INPUT_PATH)

multi = [b for b in blocks if b["n_sources"] > 1]
singoli = [b for b in blocks if b["n_sources"] == 1]

print(f"Blocchi totali: {len(blocks)}")
print(f"Da consolidare via LLM (>1 fonte): {len(multi)}")
print(f"Già singoli (1 fonte, nessuna chiamata LLM): {len(singoli)}")


# Connessione all'LLM locale

In [ ]:
from llama_index.llms.openai_like import OpenAILike

llm = OpenAILike(
    model="google/gemma-3-12b",
    api_base="http://host.docker.internal:1234/v1",
    api_key="lm-studio",
    is_chat_model=True,
    is_local=True,
    context_window=16384,
    temperature=0.2,
    timeout=300.0,  
    max_tokens=600,  
)

response = llm.complete("Ciao, funziona?")
print(response)


# Prompt di consolidamento

In [3]:
CONSOLIDATION_PROMPT = """Di seguito trovi {n} segnalazioni relative allo stesso problema ricorrente, riportate con la relativa fonte/anno.

{occorrenze}

Il tuo compito è unificarle in un'unica scheda di sintesi. Rispondi SOLO con il seguente formato, usando esattamente questi quattro campi (nessun testo prima o dopo, nessun campo aggiuntivo):

Descrizione generale: <sintesi comune del problema e del suo impatto operativo>
Causa: <causa ricorrente identificata>
Soluzione / Workaround consolidato: <cosa veniva fatto ogni volta per risolvere>
Note: <eventuali evoluzioni del problema nel tempo: se la soluzione è cambiata tra un'occorrenza e l'altra, se è mai stata trovata una soluzione definitiva>
"""


def build_occorrenze_block(sources):
    parts = []
    for i, s in enumerate(sources, start=1):
        parts.append(f"--- Occorrenza {i} ({s['source_file']}, {s['year']}) ---\n{s['text']}")
    return "\n\n".join(parts)


def consolidate_sources(sources):
    prompt = CONSOLIDATION_PROMPT.format(n=len(sources), occorrenze=build_occorrenze_block(sources))
    try:
        response = llm.complete(prompt)
    except Exception as e:
        print(f"    ERRORE chiamata LLM ({e!r}), uso fallback (concatenazione grezza, da rivedere a mano)")
        fallback = "Descrizione generale: [CONSOLIDAMENTO LLM FALLITO - rivedere manualmente]\n\n"
        fallback += build_occorrenze_block(sources)
        return fallback
    return str(response).strip()


# Elaborazione: consolido i problemi con più fonti, ripulisco gli altri

In [ ]:
def clean_singolo(text):
    lines = text.split("\n")
    lines[0] = BULLET_DATE_RE.sub("", lines[0])
    title = lines[0].strip()

    body = "\n".join(lines[1:])
    body = body.replace("*", "")  # togliamo i marcatori bold markdown
    body = body.lstrip("\n")  # togliamo la riga vuota tra titolo e Descrizione

    # uniamo il titolo dentro il campo "Descrizione generale", stessa etichetta usata dai consolidati
    body = re.sub(r"^Descrizione:\s*", f"Descrizione generale: {title}. ", body, count=1)

    return body.strip("\n") + "\n"


results = []  # {"number", "n_sources", "text"}

for i, block in enumerate(blocks, start=1):
    print(f"[{i}/{len(blocks)}] Problema {block['number']} ({block['n_sources']} fonte/i)")
    if block["n_sources"] > 1:
        final_text = consolidate_sources(block["sources"])
        print("    -> consolidato via LLM")
    else:
        final_text = clean_singolo(block["sources"][0]["text"])
        print("    -> lasciato singolo (solo pulizia intestazione, nessuna chiamata LLM)")
    results.append({"number": block["number"], "n_sources": block["n_sources"], "text": final_text})

print("Fatto.")


(64 chiamate a LLM)

# Salvataggio in Documents/singoli_dedup/

In [ ]:
output_dir = os.path.join("..", "Documents", "singoli_dedup")
os.makedirs(output_dir, exist_ok=True)

pad = len(str(len(results)))

for r in results:
    filename = f"problema_{r['number']:0{pad}d}.md"
    filepath = os.path.join(output_dir, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(r["text"])
    print(f"Scritto {filename} ({r['n_sources']} fonte/i)")

print(f"\nTotale file scritti: {len(results)} in {output_dir}")
